# Rollover Robustness + Night Session Validation
六個事先固定換月窗口 × regular/afterhours × 1!/old/new；共 36 個 sequential logical historical.candles requests。
只查 1m。保留 raw、metadata、13:45 特殊 bar；不補分鐘、不調整價格、不計算跨換月績效。
夜盤以具來源 session windows 歸屬次一 regular trade_date，另存 calendar_date。
GitHub clean branch 為 provenance；每次寫入新的 Drive timestamp_commit archive。
Private trigger 名稱未知不阻擋 sampled construction 的 operational traceability；未解釋的混約或 active asymmetric gaps 會阻擋。
此 Notebook 不呼叫 historical phase 1→7 或其他 rollover runner。

## Environment / Git / Secrets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess, sys, tempfile
from google.colab import userdata
REPO_URL = "https://github.com/hh4832/fubon_futures_intraday_data_validation.git"
REPO_DIR = Path('/content/fubon_futures_intraday_data_validation_robustness')
BRANCH = 'feature/rollover-robustness-validation'
# Public clone first; only a private repo needs optional GITHUB_TOKEN.
if not REPO_DIR.exists():
    result = subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO_DIR)],capture_output=True,text=True)
    if result.returncode:
        try:
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            raise RuntimeError('Public clone failed; private repositories need GITHUB_TOKEN') from None
        if not token:
            raise RuntimeError('GITHUB_TOKEN unavailable')
        with tempfile.TemporaryDirectory() as tmp:
            askpass = Path(tmp)/'askpass.py'
            askpass.write_text('#!/usr/bin/env python3\nimport os,sys\nprint("x-access-token" if "Username" in sys.argv[1] else os.environ["COLAB_CLONE_TOKEN"])\n')
            askpass.chmod(0o700)
            env = dict(os.environ,GIT_ASKPASS=str(askpass),GIT_TERMINAL_PROMPT='0',COLAB_CLONE_TOKEN=token)
            retry = subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO_DIR)],env=env,capture_output=True)
            del env,token
            if retry.returncode:
                raise RuntimeError('Authenticated clone failed; output suppressed')
remote = subprocess.check_output(['git','remote','get-url','origin'],cwd=REPO_DIR,text=True).strip()
if remote != REPO_URL:
    raise RuntimeError('Repository origin differs from expected URL')
os.chdir(REPO_DIR)
sys.path.insert(0,str(REPO_DIR))
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)

actual_branch = subprocess.check_output(['git','branch','--show-current'],cwd=REPO_DIR,text=True).strip()
if actual_branch != BRANCH:
    raise RuntimeError('Use the rollover feature branch checkout; no automatic branch replacement')

In [ ]:
from src.archive import git_info
provenance = git_info(REPO_DIR)
print(provenance)
if provenance['git_status']:
    raise RuntimeError('Use a clean Git checkout; no auto pull or overwrite')

In [ ]:
import platform
print({'python':sys.version,'executable':sys.executable,'platform':platform.platform(),'architecture':platform.machine()})
from src.config import load_config
cfg = load_config()
from src.rollover_robustness import load_robustness_config
cfg = load_robustness_config()
print({k:cfg[k] for k in ["product","sessions","timeframe","continuous_contract"]})
display(cfg["rollover_samples"])

In [ ]:
from tqdm.auto import tqdm
from scripts.install_fubon_sdk_colab import install
for _ in tqdm(range(1),desc='SDK setup'):
    sdk_info = install(cfg['sdk_version'],cfg['sdk_download_url'],cfg['sdk_official_sha256'])

In [ ]:
import fubon_neo
from fubon_neo.sdk import FubonSDK
print("SDK import PASS")

In [ ]:
from src.fubon_auth import load_secrets, validate_certificate, login
secrets = load_secrets(userdata.get)

In [ ]:
validate_certificate(secrets)

In [ ]:
sdk, auth_info = login(secrets)
del secrets

In [ ]:
try:
    sdk.init_realtime()
    restfutopt = sdk.marketdata.rest_client.futopt
except Exception:
    raise RuntimeError('Market-data initialization failed; SDK exception suppressed') from None
from src.historical_client import read_canonical_documentation
_, documentation_info = read_canonical_documentation(cfg)
print(documentation_info)


## Fixed-window rollover validation
1. 三組 raw responses 分別保存。
2. 保留 per-bar contractMonth，找出 observed transition。
3. OHLC / volume exact comparison；average 單獨列出。
4. 比較前一天、當天、後一天的 additive / ratio adjustment evidence；不調整資料。
5. 列出 reference absence、duplicates、third month、partial-day switch、mismatches。
6. observed volume crossover 只用本窗口 minute-candle sums，不證明 provider 規則。

即使逐根吻合，單一窗口也不能直接授權多年 continuous return 計算。


## Debug/replay gates
先 pytest → 失敗 archive offline replay → 第一組 regular live smoke；均成功才允許 36-request formal run。
Replay 不呼叫 API、不覆寫 STOPPED archive。Smoke 使用同一組既定 from/to，只查 1!/202606/202607。
Replay 的 execution PASS 與資料分類分開；若資料為 FAIL/UNRESOLVED，會停止。

In [ ]:
# Gate 1: existing + regression tests; no credentials used by tests.
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)
PYTEST_PASSED = True


In [ ]:
# Gate 2: original failed archive, normalized parquet only; no API calls.
from datetime import datetime
from zoneinfo import ZoneInfo
from scripts.replay_rollover_robustness import replay_archive
FAILED_ARCHIVE = Path(cfg["drive_root"]) / "20261014_060519_d3d96794bd6b"
replay_output = Path(cfg["drive_root"]) / (datetime.now(ZoneInfo("Asia/Taipei")).strftime("%Y%m%d_%H%M%S") + "_" + provenance["git_commit"][:12] + "_replay")
_, replay_info = replay_archive(FAILED_ARCHIVE, replay_output, "202606_202607", "regular")
if replay_info["replay_status"] != "PASS" or replay_info["analysis_classification"] not in {"PASS", "PASS WITH CAVEATS"}:
    raise RuntimeError("Offline replay did not pass; stop before live smoke/formal")
print(replay_info)


In [ ]:
# Gate 3: exactly first sample regular, three requests; separate new archive.
from src.rollover_robustness_smoke import run_first_regular_smoke
if not PYTEST_PASSED:
    raise RuntimeError("Tests must pass first")
smoke_dir, smoke_info = run_first_regular_smoke(restfutopt, REPO_DIR, sdk_info, auth_info, replay_info, cfg)
print("First-sample regular smoke archive:", smoke_dir)
if smoke_info["smoke_status"] not in {"PASS", "PASS WITH CAVEATS"}:
    raise RuntimeError("Live smoke did not pass; do not start 36-request formal run")


In [ ]:
if not PYTEST_PASSED or replay_info["replay_status"] != "PASS" or smoke_info["smoke_status"] not in {"PASS", "PASS WITH CAVEATS"}:
    raise RuntimeError("Required pytest/replay/live-smoke gates have not passed")
from src.rollover_robustness import run_rollover_robustness
run_dir, summary = run_rollover_robustness(restfutopt, REPO_DIR, sdk_info, auth_info, cfg)
print(summary)


## Diagnostics / archive

In [ ]:
import pandas as pd
for filename in ["rollover_robustness_summary.csv", "rollover_afterhours_transition_summary.csv", "rollover_session_map.csv", "rollover_missing_minutes_summary.csv"]:
    display(pd.read_csv(run_dir / "diagnostics" / filename))
print(summary)


## 結論界線
Observed fact / Documented fact / Inference / Unresolved 分開標示。
契約 transition gap 不等於市場真實報酬。
同源吻合不是 external validation；夜盤與多年 methodology stability 尚未驗證。
沒有任何策略、其他日期取樣、night session 或 higher timeframe request。
